In [1]:
from pathlib import Path
import pandas as pd

# 1. 파일 불러오기
DATA_DIR = Path("../data/raw")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

# 2. 날짜 데이터 변환
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 3. 주문 상세 금액 계산 (수량 * 단가)
order_items['total_amount'] = order_items['quantity'] * order_items['unit_price']

# 4. 데이터 하나로 합치기 (order_items 기준)
df = order_items.merge(products, on='product_id', how='left')
df = df.merge(orders, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')

# 결과 5줄 확인
df.head()

,order_item_id,order_id,product_id,quantity,unit_price,total_amount,product_name,category,price,customer_id,order_date,payment_method,order_status,name,gender,age,city,signup_date
0,1,1,216,4,13600,54400,컴팩트 에세이 그린 P216,도서,16000,1121,2026-05-18,간편결제,배송중,정민진,남성,32,울산,2025-09-17
1,2,1,34,1,87000,87000,클래식 클렌징 폼 화이트 P034,뷰티,87000,1121,2026-05-18,간편결제,배송중,정민진,남성,32,울산,2025-09-17
2,3,1,187,1,93000,93000,데일리 러닝 벨트 그레이 P187,스포츠,93000,1121,2026-05-18,간편결제,배송중,정민진,남성,32,울산,2025-09-17
3,4,1,112,1,225000,225000,플러스 핸디 청소기 블루 P112,생활가전,250000,1121,2026-05-18,간편결제,배송중,정민진,남성,32,울산,2025-09-17
4,5,2,214,1,64000,64000,스마트 클렌징 폼 화이트 P214,뷰티,64000,993,2026-02-01,신용카드,환불,홍채경,남성,45,창원,2024-12-26


In [2]:
# 1. 카테고리별 매출 집계
category_summary = df.groupby('category')['total_amount'].sum().reset_index()
print("=== 카테고리별 총 매출 ===")
print(category_summary)

# 2. 월별 매출 집계
df['year_month'] = df['order_date'].dt.to_period('M')
monthly_summary = df.groupby('year_month')['total_amount'].sum().reset_index()
print("\n=== 월별 총 매출 ===")
print(monthly_summary)

=== 카테고리별 총 매출 ===
  category  total_amount
0       도서      43071600
1       문구      46005800
2     반려동물     110816200
3       뷰티     135823500
4     생활가전     457423100
5      스포츠     154676700
6       식품     116157900
7     전자기기     286750100
8       패션     313067200
9    홈인테리어     173982200

=== 월별 총 매출 ===
   year_month  total_amount
0     2025-01      87322800
1     2025-02      68504700
2     2025-03      85551000
3     2025-04     104790900
4     2025-05     105969700
5     2025-06      82491800
6     2025-07      86357700
7     2025-08      81312400
8     2025-09      98212700
9     2025-10      84413300
10    2025-11     143736400
11    2025-12     164845200
12    2026-01      83467300
13    2026-02      73580000
14    2026-03     104586100
15    2026-04     100130600
16    2026-05     108338400
17    2026-06      84942000
18    2026-07      89221300


In [3]:
# 핵심 지표 계산
total_revenue = df['total_amount'].sum()
total_orders = df['order_id'].nunique()
aov = total_revenue / total_orders if total_orders > 0 else 0

print("=== 쇼핑몰 핵심 지표 ===")
print(f"총 매출액: {total_revenue:,.0f}원")
print(f"총 주문 건수: {total_orders:,}건")
print(f"평균 주문 금액(AOV): {aov:,.0f}원")

=== 쇼핑몰 핵심 지표 ===
총 매출액: 1,837,774,300원
총 주문 건수: 6,000건
평균 주문 금액(AOV): 306,296원
